# Single-reference DSRG(n) for H2 with sparse operators

This notebook is a compact implementation of the unitary single-reference DSRG equations for H2/STO-3G. The goal is not performance; it is to show that determinant-reference normal ordering is enough to write the formalism directly in terms of sparse second-quantized operators.

The implementation follows the DSRG idea in Evangelista, *J. Chem. Phys.* **141**, 054109 (2014), arXiv:1406.0114:

\begin{align}
\bar{H}(s) &= e^{-\hat{A}(s)} \hat{H} e^{\hat{A}(s)}, \\
\hat{A}(s) &= \hat{T}(s) - \hat{T}^{\dagger}(s).
\end{align}

`DSRG_TRUNCATION_RANK` controls the many-body rank retained after every BCH commutator:

- `1` keeps scalar and one-body normal-ordered terms.
- `2` gives the usual DSRG(2) truncation and reproduces Forte's spin-orbital `MRDSRG_SO`/`LDSRG2` result for this example when `s = 0.5`.
- Higher values keep higher-rank normal-ordered terms and include excitation amplitudes up to that rank.

The amplitude update mirrors Forte's spin-orbital `MRDSRG_SO` implementation:

\begin{equation}
t_{\mu}^{\mathrm{new}} = \left(\bar{H}_{\mu} + \Delta_{\mu} t_{\mu}\right)\frac{1 - e^{-s\Delta_{\mu}^{2}}}{\Delta_{\mu}}.
\end{equation}

For this two-orbital H2 example, Brillouin's theorem leaves only the alpha/beta double excitation with a nonzero amplitude when the truncation rank is at least two.


In [ ]:
import contextlib
import io
import itertools
import math

import numpy as np

import forte2
from forte2 import Determinant, RHF, SparseState, System
from forte2 import normal_order, sparse_operator, sparse_operator_hamiltonian
from forte2.helpers import logger

# Keep tutorial output focused on the sparse DSRG data.
logger.set_verbosity_level(0)

SCREEN = 1.0e-12
FLOW_PARAM = 5.0
DSRG_TRUNCATION_RANK = 2


## Build the H2 Hamiltonian

Forte2 supplies the Hartree-Fock orbitals and the molecular integrals. We then build an ordinary `SparseOperator` Hamiltonian in the spatial-orbital convention used by `sparse_operator_hamiltonian`.


In [ ]:
def build_h2_sparse_hamiltonian(r=0.74, basis="sto-3g"):
    xyz = f"""
    H 0.0 0.0 0.0
    H 0.0 0.0 {r}
    """

    with contextlib.redirect_stdout(io.StringIO()):
        system = System(
            xyz=xyz,
            basis_set=basis,
            minao_basis_set=None,
            cholesky_tei=True,
            cholesky_tol=1.0e-12,
        )
        rhf = RHF(charge=0, e_tol=1.0e-12, d_tol=1.0e-10)(system)
        rhf.run()

    C = rhf.C[0]
    hcore_mo = np.einsum("pq,pi,qj->ij", system.ints_hcore(), C, C, optimize=True)
    eri_mo = system.fock_builder.two_electron_integrals_block(C)
    ham = sparse_operator_hamiltonian(system.nuclear_repulsion, hcore_mo, eri_mo)
    return system, rhf, ham, np.array(rhf.eps[0])


system, rhf, ham, eps = build_h2_sparse_hamiltonian()
reference = Determinant("20")

print(f"RHF energy:         {rhf.E:.15f} Eh")
print(f"Orbital energies:   {eps}")
print(f"Reference determinant: {reference.str(rhf.nmo)}")
print(f"DSRG truncation rank: {DSRG_TRUNCATION_RANK}")


## Operator bookkeeping

A `NormalOrderedString` may differ from the physical excitation string by a phase. The helpers below keep that phase so that coefficients extracted from a `NormalOrderedSparseOperator` are converted back to the coefficient of the corresponding physical excitation operator.

The excitation generator is now rank-generic. It enumerates spin-conserving particle-hole excitations from rank 1 through `max_excitation_rank`, capped by the number of occupied and virtual spin orbitals available.


In [ ]:
def normal_key_and_phase(spop, ref):
    no_op = normal_order(spop, ref, SCREEN)
    items = [(term, coeff) for term, coeff in no_op if abs(coeff) > 1.0e-10]
    if len(items) != 1:
        raise RuntimeError(
            "Expected one normal-ordered term, got "
            + str([(term.str(ref), coeff) for term, coeff in items])
        )
    return items[0]


def physical_coeff(no_op, key, phase):
    return no_op.coefficient(key) / phase


def canonical_excitation_string(cre_modes, ann_modes):
    def token(mode, creation):
        orbital, spin = mode
        return f"{orbital}{spin}{'+' if creation else '-'}"

    alpha_cre = sorted([m for m in cre_modes if m[1] == "a"], key=lambda x: x[0])
    beta_cre = sorted([m for m in cre_modes if m[1] == "b"], key=lambda x: x[0])
    beta_ann = sorted([m for m in ann_modes if m[1] == "b"], key=lambda x: x[0], reverse=True)
    alpha_ann = sorted([m for m in ann_modes if m[1] == "a"], key=lambda x: x[0], reverse=True)

    tokens = [token(m, True) for m in alpha_cre]
    tokens += [token(m, True) for m in beta_cre]
    tokens += [token(m, False) for m in beta_ann]
    tokens += [token(m, False) for m in alpha_ann]
    return "[" + " ".join(tokens) + "]"


def enumerate_spin_conserving_excitations(nspatial, nocc, eps, ref, max_excitation_rank):
    occ = [(i, spin) for i in range(nocc) for spin in ("a", "b")]
    virt = [(a, spin) for a in range(nocc, nspatial) for spin in ("a", "b")]
    highest_rank = min(max_excitation_rank, len(occ), len(virt))
    excitations = []

    for rank in range(1, highest_rank + 1):
        for ann in itertools.combinations(occ, rank):
            ann_spins = sorted(spin for _, spin in ann)
            for cre in itertools.combinations(virt, rank):
                if ann_spins != sorted(spin for _, spin in cre):
                    continue
                label = canonical_excitation_string(cre, ann)
                op = sparse_operator(label, 1.0)
                key, phase = normal_key_and_phase(op, ref)
                denom = sum(eps[i] for i, _ in ann) - sum(eps[a] for a, _ in cre)
                excitations.append(
                    {
                        "rank": rank,
                        "label": label,
                        "op": op,
                        "key": key,
                        "phase": phase,
                        "denom": denom,
                    }
                )

    return excitations


excitations = enumerate_spin_conserving_excitations(
    rhf.nmo,
    rhf.na,
    eps,
    reference,
    max_excitation_rank=DSRG_TRUNCATION_RANK,
)
ham_no = normal_order(ham, reference, SCREEN, max_rank=DSRG_TRUNCATION_RANK)

print(f"Number of excitation amplitudes: {len(excitations)}")
for ex in excitations:
    h_mu = physical_coeff(ham_no, ex["key"], ex["phase"])
    print(
        f"rank {ex['rank']}  {ex['label']:24s}  "
        f"Delta = {ex['denom']: .12f}  H_mu = {h_mu.real: .12f}"
    )


## DSRG machinery

The BCH series is evaluated recursively. After every commutator, the result is converted to determinant-normal-ordered form and truncated with `normal_order(..., max_rank=truncation_rank)`. This is the step that defines the DSRG(n) truncation used below.

The cluster operator `T` is built explicitly as a `NormalOrderedSparseOperator`; it is expanded back to `SparseOperator` only to form `A = T - T^\dagger` for the current commutator machinery.


In [ ]:
def make_normal_ordered_cluster_operator(excitations, amplitudes, ref):
    T_no = forte2.NormalOrderedSparseOperator(ref)
    for ex, amp in zip(excitations, amplitudes):
        if abs(amp) > SCREEN:
            T_no.add(ex["key"], complex(amp) * ex["phase"])
    return T_no


def truncate_sparse_operator(op, ref, truncation_rank):
    return normal_order(op, ref, SCREEN, max_rank=truncation_rank).to_sparse_operator(SCREEN)


def sparse_operator_norm(op):
    return math.sqrt(sum(abs(coeff) ** 2 for _, coeff in op)) if len(op) else 0.0


def bch_hbar_dsrg(ham, A, ref, truncation_rank, max_comm=20, comm_thresh=1.0e-12):
    hbar = normal_order(ham, ref, SCREEN, max_rank=truncation_rank).to_sparse_operator(SCREEN)
    nested = hbar
    commutator_norms = []

    for ncomm in range(1, max_comm + 1):
        nested = nested.commutator(A)
        nested = truncate_sparse_operator(nested, ref, truncation_rank=truncation_rank)
        contribution = nested * (1.0 / math.factorial(ncomm))
        hbar += contribution

        norm = sparse_operator_norm(contribution)
        commutator_norms.append(norm)
        if norm < comm_thresh:
            break

    return normal_order(hbar, ref, SCREEN, max_rank=truncation_rank), commutator_norms


def regularized_denominator(denom, s):
    return (1.0 - math.exp(-s * denom * denom)) / denom


def solve_sparse_dsrg(
    ham,
    ref,
    excitations,
    truncation_rank,
    flow_param=0.5,
    max_iter=80,
    e_conv=1.0e-10,
    r_conv=1.0e-5,
    max_comm=20,
):
    ham_no = normal_order(ham, ref, SCREEN, max_rank=truncation_rank)
    h0 = np.array(
        [physical_coeff(ham_no, ex["key"], ex["phase"]) for ex in excitations],
        dtype=complex,
    )

    amplitudes = np.array(
        [h0[k] * regularized_denominator(ex["denom"], flow_param) for k, ex in enumerate(excitations)],
        dtype=complex,
    )

    identity_key, identity_phase = normal_key_and_phase(sparse_operator("[]", 1.0), ref)
    history = []
    previous_energy = None

    for iteration in range(max_iter + 1):
        T_no = make_normal_ordered_cluster_operator(excitations, amplitudes, ref)
        T = T_no.to_sparse_operator(SCREEN)
        A = T - T.adjoint()
        hbar_no, commutator_norms = bch_hbar_dsrg(
            ham,
            A,
            ref,
            truncation_rank=truncation_rank,
            max_comm=max_comm,
        )
        energy = physical_coeff(hbar_no, identity_key, identity_phase).real
        hbar_offdiag = np.array(
            [physical_coeff(hbar_no, ex["key"], ex["phase"]) for ex in excitations],
            dtype=complex,
        )

        new_amplitudes = np.array(
            [
                (hbar_offdiag[k] + ex["denom"] * amplitudes[k])
                * regularized_denominator(ex["denom"], flow_param)
                for k, ex in enumerate(excitations)
            ],
            dtype=complex,
        )

        update = new_amplitudes - amplitudes
        rms_update = float(np.linalg.norm(update))
        delta_energy = 0.0 if previous_energy is None else energy - previous_energy
        history.append(
            {
                "iteration": iteration,
                "energy": energy,
                "delta_energy": delta_energy,
                "rms_update": rms_update,
                "amplitudes": amplitudes.copy(),
                "hbar_offdiag": hbar_offdiag.copy(),
                "ncomm": len(commutator_norms),
            }
        )

        if previous_energy is not None and abs(delta_energy) < e_conv and rms_update < r_conv:
            return energy, amplitudes, history, hbar_no

        amplitudes = new_amplitudes
        previous_energy = energy

    raise RuntimeError(f"DSRG({truncation_rank}) iterations did not converge")


## Solve and inspect convergence

For H2/STO-3G, ranks above two have no additional amplitudes because only two electrons and two spatial orbitals are available. In larger examples, increasing `DSRG_TRUNCATION_RANK` retains higher-rank normal-ordered BCH terms and includes corresponding higher-rank particle-hole amplitudes.


In [ ]:
E_dsrg, amplitudes, history, hbar_no = solve_sparse_dsrg(
    ham,
    reference,
    excitations,
    truncation_rank=DSRG_TRUNCATION_RANK,
    flow_param=FLOW_PARAM,
)

print(f"Converged DSRG({DSRG_TRUNCATION_RANK}) energy: {E_dsrg:.15f} Eh")
print(f"Number of iterations:          {len(history)}")
print()
print("Final amplitudes")
for ex, amp in zip(excitations, amplitudes):
    print(f"  rank {ex['rank']}  {ex['label']:24s} {amp.real: .12f}")
print()
print("Last five DSRG iterations")
for row in history[-5:]:
    print(
        f"{row['iteration']:3d}  E = {row['energy']: .15f}  "
        f"dE = {row['delta_energy']: .3e}  "
        f"||dT|| = {row['rms_update']: .3e}  "
        f"ncomm = {row['ncomm']}"
    )


## DSRG(n) rank sweep

The same solver can be run for several truncation levels. For H2/STO-3G the excitation manifold contains only singles and one double excitation. Rank 3 changes the BCH truncation relative to DSRG(2), while rank 4 adds no further contribution for this minimal system.


In [ ]:
rank_sweep = []
for rank in (2, 3, 4):
    rank_excitations = enumerate_spin_conserving_excitations(
        rhf.nmo,
        rhf.na,
        eps,
        reference,
        max_excitation_rank=rank,
    )
    rank_energy, rank_amplitudes, rank_history, _ = solve_sparse_dsrg(
        ham,
        reference,
        rank_excitations,
        truncation_rank=rank,
        flow_param=FLOW_PARAM,
    )
    rank_sweep.append(
        {
            "rank": rank,
            "energy": rank_energy,
            "iterations": len(rank_history),
            "n_amplitudes": len(rank_excitations),
            "max_abs_amplitude": max(abs(amp) for amp in rank_amplitudes),
        }
    )

print(f"{'n':>3s} {'E_DSRG(n) / Eh':>22s} {'iterations':>12s} {'amplitudes':>12s} {'max |T|':>12s}")
for row in rank_sweep:
    print(
        f"{row['rank']:3d} {row['energy']:22.15f} "
        f"{row['iterations']:12d} {row['n_amplitudes']:12d} {row['max_abs_amplitude']:12.8f}"
    )


## Compare against exact diagonalization in Forte2

DSRG is not variational, so it is allowed to lie slightly below FCI for this small example. The FCI number is still useful as a scale for the correlation energy.


In [ ]:
def fci_energy_from_sparse_operator(ham, nmo, na, nb):
    dets = forte2.hilbert_space(nmo, na, nb)
    hmat = np.zeros((len(dets), len(dets)), dtype=complex)

    for j, ket_det in enumerate(dets):
        hket = ham @ SparseState({ket_det: 1.0})
        for i, bra_det in enumerate(dets):
            hmat[i, j] = forte2.overlap(SparseState({bra_det: 1.0}), hket)

    return np.linalg.eigvalsh(hmat)[0].real


E_fci = fci_energy_from_sparse_operator(ham, rhf.nmo, rhf.na, rhf.nb)
print(f"RHF energy:              {rhf.E:.15f} Eh")
print(f"FCI energy:              {E_fci:.15f} Eh")
print(f"DSRG({DSRG_TRUNCATION_RANK}) energy:        {E_dsrg:.15f} Eh")


## Forte reference check

The external Forte calculation used the spin-orbital `MRDSRG_SO` implementation with zero active orbitals:

```python
set forte {
  job_type               newdriver
  active_space_solver    detci
  correlation_solver     mrdsrg_so
  corr_level             ldsrg2
  frozen_docc            [0]
  restricted_docc        [1]
  active                 [0]
  restricted_uocc        [1]
  dsrg_s                 0.5  # Forte reference below is for s = 0.5
  dsrg_rsc_ncomm         20
  dsrg_rsc_threshold     1.0e-12
  dsrg_trans_type        unitary
  mcscf_reference        false
}
```

For H2/STO-3G at 0.74 Å and `s = 0.5`, Forte reports `-1.137730020025723 Eh` for the rank-2 LDSRG(2) truncation. The assertion is only applied when `DSRG_TRUNCATION_RANK == 2` and `FLOW_PARAM == 0.5`.


In [ ]:
forte_rank2_reference_energy = -1.1377300200257232

diff = E_dsrg - forte_rank2_reference_energy
print(f"Sparse DSRG({DSRG_TRUNCATION_RANK}): {E_dsrg:.15f} Eh")
print(f"Forte DSRG(2), s=0.5: {forte_rank2_reference_energy:.15f} Eh")
print(f"Difference:       {diff:.3e} Eh")

if DSRG_TRUNCATION_RANK == 2 and abs(FLOW_PARAM - 0.5) < 1.0e-12:
    assert abs(diff) < 5.0e-10
else:
    print("Reference assertion skipped because the Forte reference is for rank 2 at s = 0.5.")


## Linear H-chain scaling at `s = 5`

The same sparse proof-of-concept solver was applied to closed-shell linear H chains, `H_N`, with equally spaced atoms at 0.74 Angstrom in STO-3G. Each DSRG(n) solve was run in a separate process with a 305 s cutoff; a timeout marks the first system size that exceeded the 5 minute target for that truncation level. The timings below are solve times for the DSRG iteration/BCH loop and do not include the small RHF/integral setup cost.

| system | DSRG(2) | DSRG(3) | DSRG(4) |
| --- | ---: | ---: | ---: |
| H4 | 0.93 s | 3.49 s | 6.26 s |
| H6 | 17.94 s | 306.82 s | >305 s |
| H8 | 212.73 s | not run after H6 cutoff | not run after H6 cutoff |
| H10 | >305 s | not run | not run |

The first measured systems exceeding 5 minutes are therefore H10 for DSRG(2), H6 for DSRG(3), and H6 for DSRG(4). The H6 DSRG(3) calculation was rerun without the cutoff and converged just past the threshold. The H8 DSRG(2) case still completes under the cutoff, but it is already about 3.55 minutes, so the H10 timeout is a sharp practical boundary for this unoptimized sparse prototype.

Reference energies from the same sparse Hamiltonians are:

| system | RHF / Eh | FCI / Eh | DSRG(2) / Eh | DSRG(3) / Eh | DSRG(4) / Eh |
| --- | ---: | ---: | ---: | ---: | ---: |
| H4 | -2.097899614124902 | -2.138889914923100 | -2.140264303594847 | -2.138895129861726 | -2.138889864326098 |
| H6 | -3.079941945259280 | -3.142365503005163 | -3.144610080817317 | -3.142386244446432 | not converged |
